# Unit 7 — Regression Assumptions and Diagnostics

Unit 6 fitted models and read them. This unit asks whether to believe them.
Every number a regression prints rests on assumptions about the data, and most
of those assumptions are about the **residuals** — so they are checked after
the model is fitted, not before.

One rule runs through the whole unit: a check either finds evidence of a
problem or finds none. Finding none is not proof that there is none, and a
diagnostic report says which.

Four parts:

1. What "controlling for" a third variable means, as a partial correlation.
2. Model 1 of perceived stress, diagnosed end to end.
3. Models 2 and 3, compared with it on the same rows.
4. Independence, on data where it fails.

In [ ]:
import math9102 as m9
import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

m9.use_house_style()

---

## Part 1 — Controlling for a third variable

The wellbeing survey from Units 4 and 5 carries three scales used all unit:

- `tpstress`, the Perceived Stress Scale;
- `tpcoiss`, the Perceived Control of Internal States Scale;
- `tmarlow`, the Marlowe–Crowne Social Desirability Scale — how inclined a
  respondent is to answer in the way that looks good.

People who feel in control tend to feel less stressed. But if the people who
like to look good also claim calm and claim control, part of that correlation
is the habit, not the two things measured. A **partial correlation** removes
what the third variable predicts from *both* variables, then correlates what is
left.

In [ ]:
survey = m9.load_survey()

# The variables first, then the rows complete on those.
model_vars = ["tpstress", "tpcoiss", "tmarlow"]
missing = m9.missingness(survey, model_vars)
print(missing.attrs["summary"])
rows = survey[model_vars].dropna()

In [ ]:
result = pg.partial_corr(rows, x="tpcoiss", y="tpstress", covar="tmarlow")
result

In [ ]:
m9.report_partial_correlation(rows, "tpcoiss", "tpstress", "tmarlow", result,
                              x_label="perceived control", y_label="perceived stress",
                              covar_label="social desirability")

The partial correlation is barely different from the zero-order one: social
desirability accounts for little of the relationship.

### Partial is not semi-partial

A **semi-partial** correlation removes the third variable from only *one* of
the two, so it answers a different question and gives a different number.
Removed from stress, it asks whether control goes with the part of stress that
social desirability does not predict. Removed from control, it asks whether the
part of control that social desirability does not predict goes with stress.

In [ ]:
pd.DataFrame({
    "r": [np.corrcoef(rows.tpcoiss, rows.tpstress)[0, 1],
          result["r"].iloc[0],
          pg.partial_corr(rows, "tpcoiss", "tpstress", y_covar="tmarlow")["r"].iloc[0],
          pg.partial_corr(rows, "tpcoiss", "tpstress", x_covar="tmarlow")["r"].iloc[0]],
}, index=["zero-order", "partial: removed from both",
          "semi-partial: removed from stress", "semi-partial: removed from control"])

**Name the one you computed.** R's `ppcor::spcor.test` computes a semi-partial
correlation, and a published report of a "partial correlation" between these
two scales quoted its value. The report helper recomputes the partial correlation and
refuses anything else:

In [ ]:
semi = pg.partial_corr(rows, "tpcoiss", "tpstress", y_covar="tmarlow")
try:
    m9.report_partial_correlation(rows, "tpcoiss", "tpstress", "tmarlow", semi)
except ValueError as refusal:
    print(refusal)

---

## Part 2 — Model 1, diagnosed end to end

> **Model 1:** does perceived control predict perceived stress, controlling for
> social desirability?

### Select the variables, then drop rows

`rows` above already holds Model 1's complete cases. Compare that with dropping
every row that has a gap anywhere in the survey's columns:

In [ ]:
print(f"rows complete on every column: {len(survey.dropna())} of {len(survey)}")
print(f"rows complete on Model 1's variables: {len(rows)}")

In [ ]:
# Which columns cost the most rows when every column has to be complete?
m9.missingness(survey).head(5)

A row missing a variable your model never uses is not missing for your model.
Selecting the model's variables first keeps every respondent the model can use.

### How many cases?

Rules of thumb differ, and many circulate without a source. Green (1991)
suggests at least fifty plus eight per predictor to test the whole model, and a
hundred and four plus one per predictor to test each coefficient — both for a
medium-sized relationship. A smaller effect needs more.

In [ ]:
m = 2   # predictors in Model 1
print(f"Green's floors: {50 + 8 * m} for the model, {104 + m} for each coefficient; "
      f"Model 1 has {len(rows)}")

### Fit, and look at the variables

In [ ]:
model1 = smf.ols("tpstress ~ tpcoiss + tmarlow", data=rows).fit()
print(model1.summary())

In [ ]:
# Non-zero variance: every variable in the model varies.
rows.var()

Nothing above is believed yet. The rest of this part checks it.

### Residuals, visually first

The assumptions that matter most are about the residuals: the part of each
score the model did not predict. The four standard plots, on Model 1 alone:

In [ ]:
m9.residual_panel(model1);

Read them in order:

- **Residuals against fitted values:** a shapeless band about zero — no curve,
  no funnel.
- **Q–Q plot:** close to the line through the middle; both tails stray a little.
- **Scale–location:** a flat band — no sign that the spread changes along the
  line.
- **Residuals against leverage:** no case is both far out in the predictors and
  far from the line.

### Normality of the residuals: what the test adds

In [ ]:
m9.check_normality(model1)

Shapiro–Wilk is significant. With four hundred residuals it detects departures
far too small to matter, so its flag is left empty: the Q–Q plot is the
evidence. The outcome itself does not need to be normal — only the error around
the line.

### Standardised residuals

A standardised residual divides each residual by **its own** standard error,
*s*√(1 − *h*ᵢᵢ), which shrinks for a case far out in the predictors. Dividing
every residual by one standard deviation ignores that.

In [ ]:
infl = model1.get_influence()
standardised = infl.resid_studentized_internal
naive = model1.resid / model1.resid.std()
print(f"largest leverage: {infl.hat_matrix_diag.max():.3f}")
pd.DataFrame({"standardised": [standardised.min(), standardised.max()],
              "e / sd(e)": [naive.min(), naive.max()]}, index=["min", "max"])

Here the two ranges are close. Even the largest leverage, printed above the
table, leaves √(1 − *h*ᵢᵢ) near one, so no case has leverage high enough to
separate the two. With one that did, the shortcut would understate exactly the
residual that matters.

In [ ]:
print(f"beyond ±1.96: {(np.abs(standardised) > 1.96).mean():.1%} (about 5% expected)")
print(f"beyond ±3.29: {(np.abs(standardised) > 3.29).sum()} cases")

### The Bonferroni outlier test

This test uses **studentised** residuals. A studentised residual is a
standardised one whose *s* comes from the model fitted without that case, so a
case far from the line cannot inflate its own denominator and hide. The most
extreme is tested, with its *p*-value multiplied by the number of cases —
because it was picked as the largest.

In [ ]:
tests = model1.outlier_test()
tests.sort_values("student_resid", key=abs, ascending=False).head(3)

The index is the row label in the survey, so the case can be looked up:

In [ ]:
# The most extreme case: the largest studentised residual in size.
worst = tests["student_resid"].abs().idxmax()
scales = ["tpstress", "tpcoiss", "tmarlow"]
pd.DataFrame({f"row {worst}": survey.loc[worst, scales],
              "lowest": survey[scales].min(), "median": survey[scales].median(),
              "highest": survey[scales].max()})

Nothing marks these scores as an entry error: each is a whole number inside the
range the survey's respondents gave, and none sits at either end of it. The
model fits this respondent badly; nothing in the data says the answers are
wrong.

### Influence: Cook's distance and standardised DFBETAS

An outlier matters when it moves the model. **Cook's distance** measures how
far all the fitted values move when a case is left out; **standardised
DFBETAS** measure how far each coefficient moves, in its own standard errors.
Both use 1 as the usual convention, and the DFBETAS threshold belongs to the
standardised version only. Conventions differ: Belsley, Kuh and Welsch's
size-adjusted cut-off for DFBETAS, 2/√*n*, shrinks as the sample grows, and at
this size it lists many cases. Treat such a list as cases to look at, not as an
alarm.

In [ ]:
m9.check_influence(model1, top=5)

In [ ]:
k = len(model1.params)   # coefficients, counting the intercept
cut = 2 * k / len(rows)
print(f"2k/n = {cut:.3f}: {(infl.hat_matrix_diag > cut).sum()} cases above it")

Nothing comes near either alarm. Many cases have leverage above 2*k*/*n*, twice
the average leverage, where *k* counts the model's coefficients with the
intercept. They sit further out in the predictors than most, and none of them
moves the line much: the one the table above flags for leverage has a Cook's
distance far below 1.

### What to do with an outlier: run it twice

In [ ]:
without = smf.ols("tpstress ~ tpcoiss + tmarlow", data=rows.drop(index=worst)).fit()
pd.DataFrame({"with": model1.params, "without": without.params,
              "p with": model1.pvalues, "p without": without.pvalues})

In [ ]:
print(f"R squared with: {model1.rsquared:.3f}, without: {without.rsquared:.3f}")

The coefficients barely move and no conclusion changes, so the case stays and
both fits can be reported. *R²* rises without it — which is why a rule that
deletes every badly fitted case makes a model look better than it is.

**Excluding a group is a different decision.** Cases outside the population
your question is about — students who never sat an exam, in a model of exam
grades — can be set aside, but before fitting, with the reason stated and the
number shown. That is a decision about the question. Dropping cases because
the model fits them badly is a decision about the answer, and it is never
yours to make.

### Constant variance

In [ ]:
m9.check_homoscedasticity(model1)

This is Koenker's version of the Breusch–Pagan test, which does not assume
normal residuals. R's `performance` package runs the original version on the
fitted values; it gives a different *p*-value, and neither finds evidence of
changing variance:

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

original = het_breuschpagan(model1.resid, sm.add_constant(model1.fittedvalues), robust=False)
print(f"original Breusch–Pagan on the fitted values: LM = {original[0]:.2f}, p = {original[1]:.3f}")

No evidence is not proof. The residuals-against-fitted plot showed no funnel,
and that is the second half of the evidence.

### Where variance does change: a remedy

Professors' salaries, against years since the PhD. The plots first:

In [ ]:
salaries = m9.load_salaries()
ols = smf.ols("salary ~ yrs_since_phd", data=salaries).fit()
m9.residual_panel(ols);

The residuals fan out as the fitted salary rises, and the scale–location points
climb from left to right: a funnel. The test agrees:

In [ ]:
m9.check_homoscedasticity(ols)

The residuals also bend: above zero in the middle of the fitted range and below
it at both ends, so a straight line in years since the PhD misses a curve as
well. That is a question about the model's form, not its standard errors, and
neither remedy below answers it. This section keeps to the funnel.

In [ ]:
hc3 = smf.ols("salary ~ yrs_since_phd", data=salaries).fit(cov_type="HC3")
pd.DataFrame({"b": ols.params, "SE, OLS": ols.bse, "SE, HC3": hc3.bse})

**Robust (HC3) standard errors** repair the reported standard errors and leave
the slope, and the question, unchanged. Heteroscedasticity never biased the slope: it made its
standard error wrong.

A transformation changes the question instead. With log salary as the outcome,
the slope is a proportion:

In [ ]:
logged = smf.ols("np.log(salary) ~ yrs_since_phd", data=salaries).fit()
print(f"each year since the PhD goes with about "
      f"{np.exp(logged.params['yrs_since_phd']) - 1:.2%} more salary")
m9.residual_panel(logged);

In [ ]:
m9.check_homoscedasticity(logged)

And here the log does not even fix the funnel. Its scale–location points still
climb, and its Breusch–Pagan statistic, though smaller than the raw model's,
still finds the variance changing. Check the residuals of the model you report,
not the one you meant to fit.

### Collinearity

In [ ]:
m9.check_collinearity(model1)

One row per term. With two predictors the two VIFs are equal by construction.
Both are close to one: control and social desirability barely overlap.

### The diagnostic report

The paragraph is generated after fitting. It needs **your** reading of the
plots, in one sentence, because the plots are the primary evidence and no
function can read them for you. Independence is described from the design, or
declared not examined.

In [ ]:
m9.report_diagnostics(
    model1,
    plots="The residuals scatter evenly about zero with no curve or funnel, "
          "and the Q-Q plot follows the line except in the tails",
    design="each respondent answered once, and the data record no grouping",
    predictor_labels={"tpcoiss": "perceived control", "tmarlow": "social desirability"},
)

It reports each check as what it found. It never says an assumption was
"confirmed" or "met".

---

## Part 3 — Models 2 and 3

> **Model 2:** adds optimism (`toptim`).
> **Model 3:** adds sex as well.

Compared models must be fitted on the same rows. Select every variable any of
the three uses, and drop missing rows once:

In [ ]:
common = survey[["tpstress", "tpcoiss", "tmarlow", "toptim", "sex"]].dropna()
print(f"{len(common)} rows, against Model 1's own {len(rows)}")
rows.index.difference(common.index)

One respondent has no optimism score. All three models are refitted on the
shared rows:

In [ ]:
m1 = smf.ols("tpstress ~ tpcoiss + tmarlow", data=common).fit()
m2 = smf.ols("tpstress ~ tpcoiss + tmarlow + toptim", data=common).fit()
m3 = smf.ols("tpstress ~ tpcoiss + tmarlow + toptim + sex", data=common).fit()

labels = {"tpcoiss": "perceived control", "tmarlow": "social desirability",
          "toptim": "optimism", "sex[T.MALES]": "sex (male against female)"}
m9.report_model_comparison(m1, m2, predictor_labels=labels)

In [ ]:
m9.report_model_comparison(m2, m3, predictor_labels=labels)

In [ ]:
pd.concat({name: m9.check_collinearity(m).set_index("predictor")["VIF"]
           for name, m in {"Model 1": m1, "Model 2": m2, "Model 3": m3}.items()}, axis=1)

Optimism and control overlap, so their VIFs rise when optimism comes in — far
below any convention for concern.

In [ ]:
m9.report_regression(m3, outcome_label="perceived stress", predictor_labels=labels)

**An extra predictor, not a moderator.** Model 3 moves the line up or down for
one sex. It does not let the effect of control differ by sex: that would need
the product term, `tpcoiss * sex`. "Sex moderates the relationship" cannot be
claimed from Model 3.

Model 3 is diagnosed the same way as Model 1:

In [ ]:
m9.residual_panel(m3);

In [ ]:
m9.report_diagnostics(
    m3,
    plots="The residual plots show no curve or funnel, and the Q-Q plot "
          "strays only in the tails",
    design="each respondent answered once, and the data record no grouping",
    predictor_labels=labels,
)

Read the Breusch–Pagan result here closely: its *p* reports as .050, on the
conventional threshold itself, and the paragraph says so and gives no verdict.
A test this close to the line is not evidence either way, and certainly not a
confirmation. The plot shows no funnel. The way to settle whether it matters
is to refit with robust standard errors and compare:

In [ ]:
robust3 = smf.ols("tpstress ~ tpcoiss + tmarlow + toptim + sex", data=common).fit(cov_type="HC3")
pd.DataFrame({"SE, OLS": m3.bse, "SE, HC3": robust3.bse,
              "p, OLS": m3.pvalues, "p, HC3": robust3.pvalues})

Every robust standard error is larger, perceived control's the most, and every
coefficient that was significant still is. Here the borderline test changes no
conclusion; report that you checked, and how.

### Collinearity that is not a problem: a categorical block

A categorical predictor's dummies are checked as one block. The Youth Cohort's
school type has six categories, and the alphabetical one, which the formula
makes the reference, is the smallest:

In [ ]:
youth = m9.load_youthcohort().dropna(subset=["s1pared", "pcschtype"])
youth["pcschtype"].value_counts()

With that reference, most of the dummies look collinear on their own:

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

school = smf.ols("gcsepts ~ s1pared + pcschtype", data=youth).fit()
pd.Series([variance_inflation_factor(school.model.exog, i)
           for i in range(1, school.model.exog.shape[1])],
          index=school.model.exog_names[1:], name="VIF per dummy")

In [ ]:
m9.check_collinearity(school)

The generalised VIF (Fox and Monette, 1992) gives one number per term, and it
does not depend on the reference. Refit with the largest school type as the
reference, and the block's row reads the same:

In [ ]:
# Categories in order of size, so the first, the reference, is the largest.
order = youth["pcschtype"].value_counts().index
reordered = youth.assign(pcschtype=pd.Categorical(youth["pcschtype"], categories=order))
largest = smf.ols("gcsepts ~ s1pared + pcschtype", data=reordered).fit()
m9.check_collinearity(largest)

The per-dummy numbers measured how small the reference group is, not
collinearity.

---

## Part 4 — Independence, where it fails

Independence is a property of how the data were collected, and no residual plot
can show it on data like the survey's. On data where the grouping is recorded,
the residuals can show it. Unit 6's pupils sit in sixty-five schools:

In [ ]:
regression = m9.load_regression()
pupils = smf.ols("normexam ~ standlrt", data=regression).fit()
by_school = regression.assign(residual=pupils.resid)

m9.icc(by_school, "residual", "school")

In [ ]:
means = by_school.groupby("school")["residual"].agg(["mean", "sem"])
off = ((means["mean"] - 1.96 * means["sem"] > 0) | (means["mean"] + 1.96 * means["sem"] < 0))
print(f"{off.sum()} of {len(means)} schools have a mean residual whose interval misses zero")

If pupils were independent, nearly every school's interval would cross zero.
Pupils in one school are more alike than the model assumes, so its standard
errors are too small. By how much depends on what is estimated. The design
effect printed above is for a **mean**: it is how many times larger a mean's
variance is than in a sample of as many independent pupils. A slope is inflated by a
different amount, which depends on how far its predictor also clusters by
school. Cluster-robust standard errors measure it directly:

In [ ]:
clustered = smf.ols("normexam ~ standlrt", data=regression).fit(
    cov_type="cluster", cov_kwds={"groups": regression["school"]})
(clustered.bse / pupils.bse).rename("how much wider, clustered")

**For the survey there is no such grouping to check.** Each person answered
once, and the data record no household, class or site. That is all the design
tells you. Say so in the report — or say independence was not examined — and
never write that it was "met".

---

## What to take from this unit

- A **partial** correlation removes the third variable from both variables; a
  **semi-partial** from one. Name the one you computed.
- Select the model's variables, then drop rows — and show the cost.
- The assumptions are about different things: linearity about the mean,
  normality and constant variance about the **residuals**, independence about
  the **design**. The outcome need not be normal.
- Read the residual plots first. At hundreds of cases a significant normality
  test is not a verdict.
- A standardised residual divides by *s*√(1 − *h*ᵢᵢ). Outliers, leverage points
  and influential cases are different things; Cook's distance and standardised
  DFBETAS measure influence.
- Run an outlier twice. Never delete a real case for being unusual; exclude a
  group only if it is outside the question's population, decided before
  fitting and stated.
- Heteroscedasticity misstates standard errors and collinearity inflates them;
  neither biases the coefficients. Robust standard errors repair the first.
- Check a categorical predictor's collinearity as one block.
- A diagnostic report says what each check found, and never more.

## Exercise

`exercise.md` takes the sleep study: count what dropping rows costs, recode a
rating stored as text, fit and diagnose two models of sleepiness, generate the
report, and criticise a regression write-up.